# ThisAbled SAFE 모듈① 최종 재학습 (Colab GPU)

정본 `configs/module1_kcelectra_final.yaml`은 수정하지 않습니다. 기존 긴급 80건 홀드아웃을 백업한 뒤 데이터 빌드가 만든 370건 파일 대신 복원합니다. 각 학습 결과는 즉시 Google Drive에 복사합니다.

Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서 아래로 실행하세요.

In [ ]:
# 0. Drive 연결 및 저장소 준비
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/threeGuineas/thisabled-ai.git'
REPO_DIR = Path('/content/thisabled-ai')
DRIVE_ROOT = Path('/content/drive/MyDrive/thisabled-safe/module1_final_retrain')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('repo:', REPO_DIR)
print('drive backup:', DRIVE_ROOT)

In [ ]:
# 1. 의존성 설치 후 런타임/GPU 검증
!pip -q install -r requirements-colab.txt
!pip -q install pandas==2.2.3 pyarrow==17.0.0 scikit-learn==1.5.2 pyyaml==6.0.2 accelerate==1.1.1 huggingface_hub

import torch, transformers
assert torch.cuda.is_available(), 'GPU가 없습니다. Colab 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
assert transformers.__version__ == '4.46.3', 'requirements-colab.txt의 transformers 고정 버전을 확인하세요.'

In [ ]:
# 2. 입력 데이터 및 정본 config 검증
import json, hashlib, collections
import pandas as pd

CONFIG = REPO_DIR / 'configs/module1_kcelectra_final.yaml'
HOLDOUT = REPO_DIR / 'data/processed/synthetic_holdout.parquet'
HOLDOUT_BACKUP = DRIVE_ROOT / 'synthetic_holdout_80.parquet'

raw_required = [
    REPO_DIR/'data/raw/kold/kold_v1.json',
    REPO_DIR/'data/raw/unsmile/unsmile_train_v1.0.tsv',
    REPO_DIR/'data/raw/unsmile/unsmile_valid_v1.0.tsv',
]
if any(not p.exists() for p in raw_required):
    print('raw 시드 데이터가 없어 공식 다운로드 스크립트를 실행합니다.')
    subprocess.run(['python', 'scripts/download_seed_datasets.py'], check=True)
required = raw_required + [CONFIG, HOLDOUT]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f'필수 파일 누락: {missing}'
assert hashlib.sha256((REPO_DIR/'data/raw/kold/kold_v1.json').read_bytes()).hexdigest() == 'c11c29b972ecb7c63936cc8daa6e0925b6b891a78b6e6b3c321dbae238cf5468'

jsonls = sorted((REPO_DIR/'data/synthetic/emergency').glob('*/*.jsonl'))
assert len(jsonls) == 15, f'합성 JSONL은 15개여야 합니다: {len(jsonls)}개'
split_counts = collections.Counter()
for p in jsonls:
    split_counts[p.stem] += sum(1 for line in p.open(encoding='utf-8') if line.strip())
assert split_counts == {'train': 800, 'val': 150, 'test': 220}, split_counts

h = pd.read_parquet(HOLDOUT)
assert len(h) == 80 and h['label'].eq(3).all(), f'정본 홀드아웃 불일치: n={len(h)}, labels={h.label.value_counts().to_dict()}'
h.to_parquet(HOLDOUT_BACKUP, index=False)
CONFIG_SHA256 = hashlib.sha256(CONFIG.read_bytes()).hexdigest()
print('synthetic split counts:', dict(split_counts))
print('holdout:', len(h), h.label.value_counts().to_dict())
print('config sha256:', CONFIG_SHA256)

In [ ]:
# 3. 시드 데이터 빌드 + 합성 train×8 병합 + 80건 홀드아웃 복원
import shutil, subprocess

subprocess.run(['python', 'scripts/build_processed_dataset.py'], check=True)
subprocess.run(['python', 'scripts/build_final_dataset.py', '--synth-repeat', '8'], check=True)
shutil.copy2(HOLDOUT_BACKUP, HOLDOUT)

train = pd.read_parquet('data/processed/train.parquet')
val = pd.read_parquet('data/processed/val.parquet')
test = pd.read_parquet('data/processed/test.parquet')
holdout = pd.read_parquet(HOLDOUT)
synth_mask = train['source'].astype(str).str.startswith('synthetic_')
assert synth_mask.sum() > 0, '합성 train이 병합되지 않았습니다.'
assert not val['source'].astype(str).str.startswith('synthetic_').any()
assert not test['source'].astype(str).str.startswith('synthetic_').any()
assert len(holdout) == 80 and holdout.label.eq(3).all()
assert set(holdout.text).isdisjoint(set(train.text)), '80건 홀드아웃이 train에 포함됐습니다.'
assert hashlib.sha256(CONFIG.read_bytes()).hexdigest() == CONFIG_SHA256, '정본 config가 변경됐습니다.'
print('train labels:', train.label.value_counts().sort_index().to_dict())
print('synthetic rows after ×8:', int(synth_mask.sum()))
print('val labels:', val.label.value_counts().sort_index().to_dict())
print('test labels:', test.label.value_counts().sort_index().to_dict())

## 4. 1차 학습 (alpha[3]=5.0)

아래 셀은 정본 config 그대로 학습합니다. 완료 후 체크포인트를 즉시 Drive에 복사합니다.

In [ ]:
# 학습 전에 이전 module1_final이 있다면 Drive로 보존
from datetime import datetime
CKPT = REPO_DIR / 'models/checkpoints/module1_final'
if CKPT.exists():
    old = DRIVE_ROOT / ('preexisting_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    shutil.copytree(CKPT, old)
    shutil.rmtree(CKPT)

subprocess.run(['python', 'scripts/train_module1.py', '--config', str(CONFIG)], check=True)
assert hashlib.sha256(CONFIG.read_bytes()).hexdigest() == CONFIG_SHA256
for name in ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']:
    assert (CKPT/name).exists(), f'체크포인트 파일 누락: {name}'
ATTEMPT = 'alpha5'
attempt_backup = DRIVE_ROOT / ATTEMPT
if attempt_backup.exists(): shutil.rmtree(attempt_backup)
shutil.copytree(CKPT, attempt_backup)
print('Drive checkpoint backup:', attempt_backup)

In [ ]:
# 5. 실측 평가: argmax + 서빙 flagged 기준. 현재 CKPT/ATTEMPT를 평가합니다.
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForSequenceClassification.from_pretrained(CKPT).cuda().eval()

def predict(df, batch_size=64):
    texts = df['text'].astype(str).tolist()
    logits = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch_size):
            x = tokenizer(texts[i:i+batch_size], padding=True, truncation=True, max_length=128, return_tensors='pt').to('cuda')
            logits.append(model(**x).logits.float().cpu())
    prob = torch.softmax(torch.cat(logits), dim=1).numpy()
    return prob.argmax(1), prob

seed_test = pd.read_parquet('data/processed/test.parquet')
holdout = pd.read_parquet(HOLDOUT)
seed_pred, seed_prob = predict(seed_test)
hold_pred, hold_prob = predict(holdout)
y_seed = seed_test.label.to_numpy()
y_hold = holdout.label.to_numpy()
risk_prob = hold_prob[:, 1:].sum(axis=1)

result = {
    'attempt': ATTEMPT,
    'config_sha256': CONFIG_SHA256,
    'seed_test_n': int(len(seed_test)),
    'seed_macro_f1_4class': float(f1_score(y_seed, seed_pred, labels=[0,1,2,3], average='macro', zero_division=0)),
    'seed_per_class': classification_report(y_seed, seed_pred, labels=[0,1,2,3], output_dict=True, zero_division=0),
    'seed_confusion_matrix': confusion_matrix(y_seed, seed_pred, labels=[0,1,2,3]).tolist(),
    'holdout_n': int(len(holdout)),
    'holdout_emergency_recall_argmax': float((hold_pred == 3).mean()),
    'holdout_flagged_rate_adult_threshold_050': float((risk_prob >= 0.50).mean()),
    'holdout_flagged_rate_minor_threshold_035': float((risk_prob >= 0.35).mean()),
}
result['success'] = bool(
    result['holdout_emergency_recall_argmax'] >= 0.75 and
    result['holdout_flagged_rate_adult_threshold_050'] >= 0.75 and
    result['holdout_flagged_rate_minor_threshold_035'] >= 0.75 and
    result['seed_macro_f1_4class'] >= 0.68
)
report_path = REPO_DIR / f'reports/validation_reports/module1/{ATTEMPT}_final_eval.json'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
shutil.copy2(report_path, DRIVE_ROOT/report_path.name)
print(json.dumps(result, ensure_ascii=False, indent=2))
print('SUCCESS:', result['success'])

## 6. 실패한 경우에만 폴백

1차가 실패하면 아래 셀의 `FALLBACK_ALPHA=4.0`으로 학습하고 다시 평가 셀을 실행합니다. 그래도 실패하면 `3.0`으로 한 번만 더 실행합니다. 정본 YAML은 건드리지 않고 `/content`의 임시 config만 사용합니다. 각 시도 전에 `ATTEMPT`가 자동 변경됩니다.

In [ ]:
# 성공했으면 실행하지 마세요. 허용값: 4.0, 이후 필요할 때 3.0
FALLBACK_ALPHA = 4.0
assert FALLBACK_ALPHA in (4.0, 3.0)
import yaml
cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
cfg['loss']['alpha'][3] = FALLBACK_ALPHA
tmp_config = Path(f'/content/module1_final_alpha{int(FALLBACK_ALPHA)}.yaml')
tmp_config.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
assert hashlib.sha256(CONFIG.read_bytes()).hexdigest() == CONFIG_SHA256

if CKPT.exists(): shutil.rmtree(CKPT)
subprocess.run(['python', 'scripts/train_module1.py', '--config', str(tmp_config)], check=True)
for name in ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']:
    assert (CKPT/name).exists(), f'체크포인트 파일 누락: {name}'
ATTEMPT = f'alpha{int(FALLBACK_ALPHA)}'
attempt_backup = DRIVE_ROOT / ATTEMPT
if attempt_backup.exists(): shutil.rmtree(attempt_backup)
shutil.copytree(CKPT, attempt_backup)
print('학습 완료:', ATTEMPT, '— 이제 바로 위 평가 셀을 다시 실행하세요.')

In [ ]:
# 7. 성공 모델 최종 검증 및 Drive 최종본 복사
assert result['success'], '성공 기준 미달입니다. 허용된 폴백을 수행하거나 결과표와 함께 중단하세요.'
assert result['holdout_n'] == 80
assert hashlib.sha256(CONFIG.read_bytes()).hexdigest() == CONFIG_SHA256
final_drive = DRIVE_ROOT / 'module1_final'
if final_drive.exists(): shutil.rmtree(final_drive)
shutil.copytree(CKPT, final_drive)
shutil.copy2(report_path, final_drive/'final_evaluation.json')

states = sorted(CKPT.glob('checkpoint-*/trainer_state.json'))
if states:
    state = json.loads(states[-1].read_text())
    eval_recalls = [x.get('eval_emergency_recall') for x in state.get('log_history', []) if 'eval_emergency_recall' in x]
    print('trainer_state eval_emergency_recall history:', eval_recalls)
    print('주의: seed val의 class 3 support가 0이면 이 값은 정의상 0입니다. 최종 판정은 80건 final_evaluation.json을 사용합니다.')
else:
    print('checkpoint 하위 trainer_state.json 없음; 최종 모델 파일과 final_evaluation.json은 검증됨')
print('FINAL DRIVE BACKUP:', final_drive)

## 8. Hugging Face 업로드 (성공 후에만)

다음 셀은 Colab 비밀키의 `HF_TOKEN`(write 권한)을 사용합니다. 토큰을 코드에 직접 적지 마세요.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
assert result['success']
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
subprocess.run([
    'hf', 'upload', 'soyuncj/thisabled-safety-kcelectra', str(CKPT), '.',
    '--repo-type', 'model', '--private',
    '--exclude', 'optimizer.pt', '--exclude', 'rng_state.pth',
    '--exclude', 'scheduler.pt', '--exclude', 'training_args.bin',
], check=True)
print('Hugging Face 업로드 완료')